In [14]:
# First, we import the 'pandas' library. Think of pandas as a powerful version of Excel that works inside Python.
# We use 'as pd' so we can type 'pd' instead of 'pandas' every time we use it.
import pandas as pd

# Here, we use pd.read_csv to tell Python to open our data file (a CSV file) and turn it into a 'DataFrame'.
# A DataFrame is just a fancy name for a table with rows and columns.
# '../' tells Python to look in the folder one level above where this notebook is located.
df = pd.read_csv('../data/raw/returns_sustainability_dataset.csv')

# We use the .shape command to see how big our table is. 
# It will print two numbers: (the number of rows, the number of columns).
print("Shape:", df.shape)

# The .head() command shows us the first 5 rows of our table.
# This is a great way to peek at the data and make sure everything looks correct.
print(df.head())

# The .dtypes command tells us what kind of information is in each column.
# 'int64' means whole numbers, 'float64' means numbers with decimals, and 'object' usually means text.
print(df.dtypes)

# Finally, we check if any information is missing. 
# .isnull() finds empty spots, and .sum() counts how many empty spots are in each column.
# Ideally, we want to see zeros everywhere, meaning our data is complete!
print(df.isnull().sum())

Shape: (5000, 23)
   Order_ID Product_ID   User_ID  Order_Date Product_Category  Product_Price  \
0  ORD00000   PROD0169  USER0195  2022-01-14         Clothing        1720.71   
1  ORD00001   PROD0318  USER1469  2022-01-03             Toys         744.06   
2  ORD00002   PROD0427  USER1812  2025-03-16         Clothing         983.68   
3  ORD00003   PROD0323  USER1274  2024-11-06            Books        1855.65   
4  ORD00004   PROD0325  USER0551  2023-06-07  Home Appliances        1770.97   

   Order_Quantity  Discount_Applied Shipping_Method Payment_Method  ...  \
0               2             30.46        Next-Day         Wallet  ...   
1               5             29.62        Next-Day         Wallet  ...   
2               5             47.80         Express         Wallet  ...   
3               2              2.90         Express            COD  ...   
4               5             44.42         Express            COD  ...   

   Return_Status Return_Reason Days_to_Return  Ord

In [15]:
# The .value_counts() command is like a quick tally. 
# Here, we are counting how many times 'Returned' and 'Not Returned' appear in the 'Return_Status' column.
print("Return Status distribution:")
print(df['Return_Status'].value_counts())

# We do the same thing for 'Return_Reason'. 
# This helps us understand the main reasons why customers are sending items back.
print("\nReturn Reason distribution:")
print(df['Return_Reason'].value_counts())

# We also count how many orders we have for each type of product (like Clothing, Electronics, etc.).
print("\nProduct Categories:")
print(df['Product_Category'].value_counts())

Return Status distribution:
Return_Status
Not Returned    3550
Returned        1450
Name: count, dtype: int64

Return Reason distribution:
Return_Reason
No Return       3550
Defective        382
Changed Mind     379
Wrong Item       348
Size Issue       341
Name: count, dtype: int64

Product Categories:
Product_Category
Clothing           1766
Electronics        1244
Books               762
Toys                752
Home Appliances     476
Name: count, dtype: int64


In [16]:
# This block of code calculates the 'Return Rate' for each product category.
# 1. df.groupby('Product_Category'): We tell Python to group our rows by their category.
# 2. .apply(lambda x: ...): For each group, we run a small calculation.
# 3. (x['Return_Status'] == 'Returned').mean(): We find the percentage of rows marked as 'Returned'.
# 4. .sort_values(ascending=False): We sort the results so the category with the highest return rate is at the top.
# 5. .round(3): We keep only 3 numbers after the decimal point to make it easier to read.
print("Return rate by category:")
category_return_rate = df.groupby('Product_Category').apply(
    lambda x: (x['Return_Status'] == 'Returned').mean()
).sort_values(ascending=False).round(3)
print(category_return_rate)

Return rate by category:
Product_Category
Clothing           0.374
Home Appliances    0.256
Electronics        0.252
Toys               0.241
Books              0.226
dtype: float64


C:\Users\vaibh\AppData\Local\Temp\ipykernel_15816\4193269424.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  category_return_rate = df.groupby('Product_Category').apply(


In [17]:
# Here we create a new, smaller table that only contains orders that were actually 'Returned'.
returned = df[df['Return_Status'] == 'Returned']

# We use 'len(df)' to count the total number of orders and 'len(returned)' for the number of returns.
print(f"Total orders: {len(df)}")
print(f"Total returned: {len(returned)}")

# We divide the number of returns by the total orders to get the overall return rate (percentage).
print(f"Overall return rate: {len(returned)/len(df):.1%}")

# Revenue impact: We add up all the values in the 'Profit_Loss' column for returned items using .sum().
print(f"\nTotal revenue lost to returns: {returned['Profit_Loss'].sum():,.0f}")

# Average cost: We find the average amount spent on processing each return using .mean().
print(f"Average return cost per order: {returned['Return_Cost'].mean():,.2f}")

# Sustainability impact: We add up the CO2 emissions and waste avoidance potential to see the environmental cost.
print(f"\nCO2 generated by returns: {returned['CO2_Emissions'].sum():,.2f}")
print(f"CO2 we could save if returns avoided: {df['CO2_Saved'].sum():,.2f}")
print(f"Waste avoided potential: {df['Waste_Avoided'].sum():,.2f}")

Total orders: 5000
Total returned: 1450
Overall return rate: 29.0%

Total revenue lost to returns: 3,079,392
Average return cost per order: 200.00

CO2 generated by returns: 2,167.50
CO2 we could save if returns avoided: 5,333.00
Waste avoided potential: 2,133.40


In [18]:
# We look for individual users who have returned many items. 
# This helps us identify 'serial returners' who might need special attention.
print("Top 10 users by return count:")
print(df[df['Return_Status'] == 'Returned']['User_ID'].value_counts().head(10))

# We check if customers using a specific payment method (like Credit Card or COD) are more likely to return items.
# We use 'include_groups=False' to keep the code running smoothly without warnings.
print("\nReturn rate by payment method:")
print(df.groupby('Payment_Method').apply(
    lambda x: (x['Return_Status'] == 'Returned').mean(),
    include_groups=False
).sort_values(ascending=False).round(3))

# We also check if the shipping speed (Standard vs. Next-Day) affects how often people return their orders.
print("\nReturn rate by shipping method:")
print(df.groupby('Shipping_Method').apply(
    lambda x: (x['Return_Status'] == 'Returned').mean(),
    include_groups=False
).sort_values(ascending=False).round(3))

Top 10 users by return count:
User_ID
USER1355    6
USER1684    5
USER1287    4
USER0932    4
USER0656    4
USER1805    4
USER0067    4
USER1694    4
USER0505    4
USER1532    4
Name: count, dtype: int64

Return rate by payment method:
Payment_Method
Credit Card    0.310
COD            0.305
Debit Card     0.275
Wallet         0.271
dtype: float64

Return rate by shipping method:
Shipping_Method
Standard    0.294
Express     0.291
Next-Day    0.285
dtype: float64


In [19]:
# Finally, we save our work! We take our DataFrame and save it as a new CSV file in the 'processed' folder.
# 'index=False' tells Python not to add an extra column for row numbers, keeping our file clean.
df.to_csv('../data/processed/returns_clean.csv', index=False)
print("Saved successfully!")

Saved successfully!
